# Interpolação

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/06-interpolacao.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. construir o polinômio que passa por $n+1$ pontos, nas formas de Lagrange e de Newton;
> 2. usar a fórmula do erro de interpolação para prever onde e quanto um interpolador erra;
> 3. explicar o fenômeno de Runge e por que os nós de Chebyshev o evitam;
> 4. usar splines cúbicas e saber quando preferi-las a um polinômio de grau alto;
> 5. distinguir interpolar dados exatos de ajustar dados com ruído.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from scipy.interpolate import CubicSpline, PchipInterpolator, BarycentricInterpolator
from cnlivro.interpolacao import diferencas_divididas, newton_avalia, lagrange, nos_chebyshev, spline_cubica_natural

## 1. Problema de engenharia: a viscosidade de um óleo

A perda de carga num mancal, o atrito numa tubulação (capítulo 3) e a potência de uma bomba dependem da viscosidade do fluido, que varia muito com a temperatura. Num ensaio de laboratório, um óleo lubrificante foi aquecido a $92\ ^\circ$C e deixado esfriar, e o viscosímetro registrou a viscosidade dinâmica $\mu$ em 13 instantes:

In [ ]:
# Dados do ensaio, na ordem em que foram registrados (o óleo esfriando)
T_ensaio = np.array([92.0, 85.5, 78.1, 71.9, 64.0, 58.2, 51.0, 45.3, 44.1, 37.6, 30.2, 24.8, 20.0])   # °C
mu_ensaio = np.array([26.1, 30.6, 36.4, 41.9, 52.1, 61.7, 78.4, 94.6, 99.5, 123.5, 172.5, 215.2, 272.1])  # mPa·s
for T, mu in zip(T_ensaio, mu_ensaio):
    print(f"T = {T:5.1f} °C    μ = {mu:6.1f} mPa·s")

Uma simulação do mancal precisa de $\mu$ em **qualquer** temperatura entre 20 e $92\ ^\circ$C, não só nas 13 medidas. Precisamos de uma função $\mu(T)$ que passe pelos pontos medidos e se comporte bem entre eles. Esse é o problema da **interpolação**.

Repare em duas coisas que vão importar: os pontos **não** são igualmente espaçados (dois deles, a 45,3 e $44{,}1\ ^\circ$C, estão quase juntos), e toda medida tem ruído, aqui de cerca de 1 %.

## 2. A ideia em uma imagem

Por $n+1$ pontos passa exatamente um polinômio de grau $n$. Parece a escolha natural. A figura mostra o que acontece ao interpolar a função $f(x) = 1/(1 + 25x^2)$, que é suave e inofensiva, com 11 pontos igualmente espaçados, com 11 pontos de Chebyshev e com uma spline cúbica.

In [ ]:
f_runge = lambda x: 1 / (1 + 25 * x**2)
t = np.linspace(-1, 1, 801)
n = 11
xe = np.linspace(-1, 1, n)
xc = nos_chebyshev(n)
casos = [("polinômio, nós iguais", xe, lagrange(xe, f_runge(xe), t)),
         ("polinômio, nós de Chebyshev", xc, lagrange(xc, f_runge(xc), t)),
         ("spline cúbica, nós iguais", xe, CubicSpline(xe, f_runge(xe))(t))]
fig = make_subplots(rows=1, cols=3, subplot_titles=[c[0] for c in casos], horizontal_spacing=0.05, shared_yaxes=True)
for j, (nome, nos, p) in enumerate(casos, start=1):
    fig.add_trace(go.Scatter(x=t, y=f_runge(t), name="f(x)", line=dict(color=CORES["exata"]), showlegend=(j == 1)), 1, j)
    fig.add_trace(go.Scatter(x=t, y=p, name="interpolador", line=dict(color=CORES["aproximacao"]), showlegend=(j == 1)), 1, j)
    fig.add_trace(go.Scatter(x=nos, y=f_runge(nos), mode="markers", name="nós", showlegend=(j == 1),
                             marker=dict(color=CORES["erro"], size=8)), 1, j)
    erro = np.abs(p - f_runge(t)).max()
    fig.add_annotation(x=0, y=-0.45, text=f"erro máximo {numero(erro, 3)}", showarrow=False, row=1, col=j,
                       font=dict(color=CORES["erro"]))
fig.update_yaxes(range=[-0.6, 2.1])
fig.update_layout(width=980, height=380, margin=dict(t=70), legend=dict(orientation="h", y=-0.15))
mostrar(fig)

**O que notar:** o polinômio com nós igualmente espaçados acerta os 11 pontos, mas oscila violentamente perto das pontas. É o **fenômeno de Runge**. Basta mudar **onde** ficam os nós, concentrando-os nas pontas (nós de Chebyshev), para o mesmo grau dar um erro pequeno. A spline troca o polinômio único por pedaços de grau 3 e também não oscila. A seção 3 explica as três coisas com uma só fórmula.

## 3. Formulação matemática

### 3.1 Existência e unicidade

Dados $n+1$ pontos $(x_0, y_0), \ldots, (x_n, y_n)$ com abscissas distintas, existe um **único** polinômio $p_n$ de grau $\le n$ com $p_n(x_i) = y_i$. Escrevendo $p_n(x) = a_0 + a_1 x + \cdots + a_n x^n$, as condições formam um sistema linear com a **matriz de Vandermonde**, $V_{ij} = x_i^j$. Ela é invertível quando os $x_i$ são distintos, mas é muito mal condicionada (seção 7.2). Por isso não se calcula o interpolador resolvendo esse sistema: usam-se as formas de Lagrange ou de Newton, que dão o **mesmo** polinômio escrito de outro jeito.

### 3.2 Forma de Lagrange

Para cada nó, o polinômio de grau $n$

$$
\ell_i(x) = \prod_{j \ne i} \frac{x - x_j}{x_i - x_j}
$$

vale 1 em $x_i$ e 0 nos outros nós. Então

$$
p_n(x) = \sum_{i=0}^{n} y_i\, \ell_i(x).
$$ (eq-lagrange-interp)

A forma de Lagrange é ótima para teoria. Para calcular, usa-se uma versão reorganizada, a **fórmula baricêntrica**, que custa $O(n)$ por ponto e é numericamente estável (`scipy.interpolate.BarycentricInterpolator`).

### 3.3 Forma de Newton e diferenças divididas

A forma de Newton escreve o polinômio de modo que acrescentar um ponto acrescenta um termo:

$$
p_n(x) = f[x_0] + f[x_0, x_1](x - x_0) + f[x_0, x_1, x_2](x - x_0)(x - x_1) + \cdots
$$

Os coeficientes são as **diferenças divididas**, definidas recursivamente:

$$
f[x_i] = y_i, \qquad
f[x_i, \ldots, x_{i+k}] = \frac{f[x_{i+1}, \ldots, x_{i+k}] - f[x_i, \ldots, x_{i+k-1}]}{x_{i+k} - x_i}.
$$

A diferença dividida de ordem $k$ é uma aproximação de $f^{(k)}/k!$. É fácil de calcular à mão (seção 4).

### 3.4 O erro de interpolação

Se $f$ tem $n+1$ derivadas contínuas, para cada $x$ existe $\xi$ no intervalo dos nós tal que

$$
f(x) - p_n(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!}\, \prod_{i=0}^{n} (x - x_i).
$$ (eq-erro-interp)

A fórmula separa o erro em duas partes:

- $f^{(n+1)}/(n+1)!$ depende da **função**. Para $1/(1+25x^2)$, as derivadas crescem tão depressa que esse fator não diminui com $n$.
- $\omega(x) = \prod (x - x_i)$ depende só dos **nós**. Com nós igualmente espaçados, $|\omega(x)|$ é enorme perto das pontas e pequeno no meio. É essa a origem do fenômeno de Runge.

### 3.5 Nós de Chebyshev

Que nós tornam $\max |\omega(x)|$ o menor possível em $[-1, 1]$? A resposta são os **nós de Chebyshev**, as projeções no eixo $x$ de pontos igualmente espaçados num semicírculo:

$$
x_k = \cos\!\left(\frac{(2k+1)\pi}{2(n+1)}\right), \qquad k = 0, \ldots, n.
$$

Eles se concentram nas pontas, exatamente onde os nós igualmente espaçados deixam $\omega$ crescer. Com eles, $\max|\omega| = 2^{-n}$, e o interpolador converge para toda função suave. Quando você **pode escolher** onde medir, ou onde avaliar uma função cara, escolha nós de Chebyshev. Num intervalo $[a, b]$, basta a mudança de variável $x \mapsto \tfrac{a+b}{2} + \tfrac{b-a}{2}x$.

### 3.6 Splines cúbicas

Quando os pontos são dados, como num ensaio, não dá para escolher os nós. A alternativa é desistir do polinômio único e usar um polinômio cúbico **em cada intervalo** $[x_i, x_{i+1}]$, colados de modo que a função, a primeira e a segunda derivadas sejam contínuas. Isso é uma **spline cúbica**, o nome da régua flexível que os desenhistas usavam para traçar curvas suaves.

As incógnitas são as segundas derivadas $M_i = S''(x_i)$. A continuidade de $S'$ em cada nó interno dá uma equação

$$
h_{i-1} M_{i-1} + 2(h_{i-1} + h_i) M_i + h_i M_{i+1} = 6\left(\frac{y_{i+1} - y_i}{h_i} - \frac{y_i - y_{i-1}}{h_{i-1}}\right),
$$

com $h_i = x_{i+1} - x_i$. Faltam duas equações, uma em cada ponta. As escolhas comuns são a spline **natural** ($M_0 = M_n = 0$), a **not-a-knot** (padrão do SciPy) e a **presa** (derivadas dadas nas pontas). O sistema é **tridiagonal** e diagonal dominante, e se resolve em $O(n)$ operações (capítulo 4).

O erro de uma spline cúbica com espaçamento $h$ é $O(h^4)$, e um ponto ruim afeta só a vizinhança, sem espalhar oscilações pelo intervalo inteiro. Se a monotonicidade importa, como numa viscosidade que só pode cair com a temperatura, o interpolador **PCHIP** (`PchipInterpolator`) sacrifica a segunda derivada contínua para nunca criar máximos ou mínimos que os dados não têm.

## 4. À mão

Para ter um valor de referência com que comparar, usamos aqui a **água**, cuja viscosidade está em qualquer livro de fenômenos de transporte. A tabela dá $\mu$ = 1,002; 0,653 e 0,467 mPa·s a 20, 40 e $60\ ^\circ$C. Estime $\mu$ a $35\ ^\circ$C com o polinômio de grau 2 que passa por esses três pontos.

**Diferenças divididas.**

| $T_i$ | $f[T_i]$ | 1ª ordem | 2ª ordem |
| --- | --- | --- | --- |
| 20 | 1,002 | | |
| | | $\dfrac{0{,}653 - 1{,}002}{40 - 20} = -0{,}01745$ | |
| 40 | 0,653 | | $\dfrac{-0{,}0093 - (-0{,}01745)}{60 - 20} = 0{,}00020375$ |
| | | $\dfrac{0{,}467 - 0{,}653}{60 - 40} = -0{,}0093$ | |
| 60 | 0,467 | | |

**Avaliação em $T = 35$.** Com os termos da forma de Newton:

$$
p_2(35) = 1{,}002 + (-0{,}01745)(35 - 20) + 0{,}00020375\,(35 - 20)(35 - 40)
= 1{,}002 - 0{,}26175 - 0{,}01528 = 0{,}7250 .
$$

O valor de tabela é 0,7195 mPa·s, então o erro é de 0,8 %. A interpolação **linear** entre 20 e 40, que é só o termo de primeira ordem, daria $0{,}7403$, com erro de 2,9 %. O termo quadrático corrigiu a curvatura.

In [ ]:
T_nos = np.array([20.0, 40.0, 60.0])
mu_nos = np.array([1.002, 0.653, 0.467])
c = diferencas_divididas(T_nos, mu_nos)
print("diferenças divididas:", c)
print("p2(35) =", newton_avalia(c, T_nos, 35.0), "  (tabela: 0,7195)")

## 5. Explore

Arraste o slider para mudar o número de nós $n$ na interpolação de $f(x) = 1/(1 + 25x^2)$. Compare os nós igualmente espaçados com os de Chebyshev. O título mostra o erro máximo de cada um.

In [ ]:
f_runge = lambda x: 1 / (1 + 25 * x**2)
t = np.linspace(-1, 1, 801)

def figura_runge(n):
    xe = np.linspace(-1, 1, n)
    xc = nos_chebyshev(n)
    pe = lagrange(xe, f_runge(xe), t)
    pc = lagrange(xc, f_runge(xc), t)
    ee, ec = np.abs(pe - f_runge(t)).max(), np.abs(pc - f_runge(t)).max()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=f_runge(t), name="f(x)", line=dict(color=CORES["exata"], width=3)))
    fig.add_trace(go.Scatter(x=t, y=pe, name="igualmente espaçados", line=dict(color=CORES["erro"])))
    fig.add_trace(go.Scatter(x=t, y=pc, name="Chebyshev", line=dict(color=CORES["aproximacao"])))
    fig.add_trace(go.Scatter(x=xe, y=np.full(n, -0.55), mode="markers", name="nós igualmente espaçados",
                             marker=dict(color=CORES["erro"], symbol="line-ns-open", size=12, line=dict(width=2))))
    fig.add_trace(go.Scatter(x=xc, y=np.full(n, -0.7), mode="markers", name="nós de Chebyshev",
                             marker=dict(color=CORES["aproximacao"], symbol="line-ns-open", size=12, line=dict(width=2))))
    fig.update_xaxes(title="x")
    fig.update_yaxes(title="y", range=[-0.8, 1.6])
    fig.update_layout(title=f"n = {n} nós: erro máximo {numero(ee, 3)} (iguais) × {numero(ec, 3)} (Chebyshev)", height=440)
    return fig

valores_n = [3, 5, 7, 9, 11, 13, 15, 19, 25]
quadros = [(str(n), figura_runge(n)) for n in valores_n]
mostrar(com_controle(quadros, rotulo="n = ", ativo=6, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com nós igualmente espaçados, o erro **cresce** com $n$: mais pontos pioram o resultado. Com nós de Chebyshev, o erro cai sem parar. Os pontinhos embaixo mostram onde ficam os nós: os de Chebyshev se acumulam nas pontas.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Tenho 13 medidas de viscosidade de um óleo em função da temperatura (vetores `T_ensaio` e `mu_ensaio`). Escreva em Python uma função `viscosidade(T)` que interpola esses dados para qualquer temperatura entre 20 e 92 °C. Escreva um teste que confira que ela reproduz as medidas."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
coeficientes = np.polyfit(T_ensaio, mu_ensaio, len(T_ensaio) - 1)   # polinômio pelos 13 pontos

def viscosidade(T):
    return np.polyval(coeficientes, T)

# Teste escrito pelo agente
erro_nos = np.abs(viscosidade(T_ensaio) - mu_ensaio).max()
print(f"maior diferença nos pontos medidos: {erro_nos:.1e} mPa·s  ->  teste {'passou' if erro_nos < 1e-2 else 'falhou'}")
print(f"viscosidade a 55 °C: {viscosidade(55.0):.1f} mPa·s")

O teste passa, e o valor a $55\ ^\circ$C parece razoável. Mas o teste só olhou os **nós**, que é exatamente onde todo interpolador acerta por construção.

### 6.2 Verificação: invariantes físicos e um gráfico

Duas propriedades físicas valem para qualquer óleo: a viscosidade é **positiva** e **diminui** quando a temperatura aumenta. Testamos as duas numa grade fina e desenhamos o resultado:

In [ ]:
T_fino = np.linspace(20, 92, 1000)
mu_agente = viscosidade(T_fino)
print(f"menor valor: {mu_agente.min():.1f} mPa·s  (positiva? {mu_agente.min() > 0})")
print(f"sempre decrescente? {np.all(np.diff(mu_agente) < 0)}")

ordem = np.argsort(T_ensaio)
spline = CubicSpline(T_ensaio[ordem], mu_ensaio[ordem])
pchip = PchipInterpolator(T_ensaio[ordem], mu_ensaio[ordem])

fig = go.Figure()
fig.add_trace(go.Scatter(x=T_fino, y=mu_agente, name="polinômio de grau 12 (agente)", line=dict(color=CORES["erro"])))
fig.add_trace(go.Scatter(x=T_fino, y=spline(T_fino), name="spline cúbica", line=dict(color=CORES["aproximacao"])))
fig.add_trace(go.Scatter(x=T_fino, y=pchip(T_fino), name="PCHIP", line=dict(color=CORES["destaque"], dash="dash")))
fig.add_trace(go.Scatter(x=T_ensaio, y=mu_ensaio, mode="markers", name="medidas", marker=dict(color=CORES["exata"], size=8)))
fig.add_hline(y=0, line=dict(color=CORES["referencia"], width=1))
fig.update_xaxes(title="temperatura T (°C)")
fig.update_yaxes(title="viscosidade μ (mPa·s)", range=[-300, 350])
fig.update_layout(title="O polinômio acerta os 13 pontos e erra entre eles")
mostrar(fig)

**O que notar:** perto de $90\ ^\circ$C, o polinômio do agente chega a **viscosidades negativas**, da ordem de $-200$ mPa·s. Entre 85 e $92\ ^\circ$C, onde o mancal trabalha, ele é inútil. A spline e o PCHIP passam pelos mesmos 13 pontos e se comportam como um óleo.

### 6.3 Por que

São três motivos, todos da seção 3:

1. **Grau alto com nós dados.** Os nós não são de Chebyshev, e o fator $\omega(x) = \prod(x - T_i)$ da fórmula {eq}`eq-erro-interp` é enorme perto das pontas.
2. **Ruído.** As medidas têm cerca de 1 % de ruído. Um polinômio de grau 12 é obrigado a passar por cada ponto, ruído incluído. O par de pontos a 44,1 e $45{,}3\ ^\circ$C é o pior caso: uma diferença de ruído entre dois pontos tão próximos vira uma inclinação falsa enorme.
3. **Condicionamento.** `np.polyfit` monta uma matriz de Vandermonde com potências de $T$ até $92^{12} \approx 4 \times 10^{23}$. Na seção 7.2 veremos o que isso faz com os coeficientes.

### 6.4 A correção

- Para **interpolar** dados medidos, use interpolação por partes: `CubicSpline` ou, quando a monotonicidade importa, `PchipInterpolator`. Ordene os dados por $T$ antes, porque as duas classes exigem abscissas crescentes.
- Como os dados têm ruído, a melhor resposta muitas vezes **não é interpolar**, e sim **ajustar** um modelo físico com poucos parâmetros. Para óleos, a equação de Vogel, $\mu = A\,e^{B/(T - C)}$, é o modelo usual. Esse é o assunto do capítulo 7.
- Em qualquer caso, teste **entre** os nós: invariantes físicos, como sinal e monotonicidade, e a validação cruzada, que consiste em retirar um ponto, interpolar sem ele e comparar.

In [ ]:
def validacao_cruzada(construtor):
    erros = []
    for i in range(1, len(T_ensaio) - 1):                  # retira um ponto interno por vez
        mascara = np.arange(len(T_ensaio)) != i
        Ts, mus = T_ensaio[mascara], mu_ensaio[mascara]
        o = np.argsort(Ts)
        erros.append(abs(construtor(Ts[o], mus[o])(T_ensaio[i]) - mu_ensaio[i]) / mu_ensaio[i])
    return 100 * np.max(erros)

print(f"pior erro ao prever o ponto retirado: polinômio {validacao_cruzada(lambda x, y: np.poly1d(np.polyfit(x, y, len(x) - 1))):.0f} %,"
      f"  spline {validacao_cruzada(CubicSpline):.1f} %,  PCHIP {validacao_cruzada(PchipInterpolator):.1f} %")

> **Lição.** Todo interpolador acerta os nós. Um teste que só confere os nós não testa nada. Teste entre os nós, com invariantes físicos e validação cruzada.

## 7. Quando falha

### 7.1 Extrapolação

Fora do intervalo dos dados, nenhum interpolador tem garantia. O fator $\omega(x)$ de {eq}`eq-erro-interp` cresce como $|x|^{n+1}$ fora dos nós, e as splines continuam o último cúbico, que pode ir para qualquer lado:

In [ ]:
for T in [100.0, 110.0, 120.0]:
    print(f"T = {T:5.1f} °C:  spline {spline(T):8.1f}   PCHIP {pchip(T):6.1f}   polinômio {viscosidade(T):12.1f}  mPa·s")

A viscosidade real a $120\ ^\circ$C é da ordem de 15 mPa·s. Para extrapolar, só um **modelo físico** ajustado aos dados, como a equação de Vogel, tem alguma chance, e mesmo assim com cautela.

### 7.2 A matriz de Vandermonde

Calcular o interpolador resolvendo o sistema de Vandermonde é a forma mais direta e a pior. O número de condição cresce exponencialmente com o grau, mesmo em $[-1, 1]$:

In [ ]:
graus = np.arange(2, 31, 2)
cond_igual = [np.linalg.cond(np.vander(np.linspace(-1, 1, n + 1))) for n in graus]
cond_cheb = [np.linalg.cond(np.vander(nos_chebyshev(n + 1))) for n in graus]
graus_ensaio = [n for n in graus if n < len(T_ensaio)]
cond_ensaio = [np.linalg.cond(np.vander(T_ensaio[:n + 1])) for n in graus_ensaio]
fig = go.Figure()
fig.add_trace(go.Scatter(x=graus, y=cond_igual, mode="lines+markers", name="nós iguais em [−1, 1]", line=dict(color=CORES["erro"])))
fig.add_trace(go.Scatter(x=graus, y=cond_cheb, mode="lines+markers", name="nós de Chebyshev em [−1, 1]", line=dict(color=CORES["aproximacao"])))
fig.add_trace(go.Scatter(x=graus_ensaio, y=cond_ensaio, mode="lines+markers", name="temperaturas do ensaio (°C)", line=dict(color=CORES["secundaria"])))
fig.add_hline(y=1 / np.finfo(float).eps, line=dict(color=CORES["referencia"], dash="dash"),
              annotation_text="1/ε_mach: nenhum algarismo correto", annotation_position="top left")
fig.update_xaxes(title="grau n")
fig.update_yaxes(type="log", title="κ(V)", exponentformat="power")
fig.update_layout(title="O número de condição da matriz de Vandermonde explode")
mostrar(fig)

**O que notar:** mesmo com nós de Chebyshev, $\kappa(V)$ cresce exponencialmente, porque a base $1, x, x^2, \ldots$ é ruim: as potências altas ficam quase paralelas. Com as temperaturas do ensaio em graus Celsius, o problema é pior ainda, porque as colunas têm escalas que vão de 1 a $10^{23}$. As formas de Lagrange baricêntrica e de Newton não montam essa matriz, e por isso funcionam.

### 7.3 Dados fora de ordem

Muitas rotinas de interpolação exigem abscissas crescentes. O ensaio registrou as temperaturas em ordem **decrescente**, porque o óleo esfriava. `CubicSpline` acusa o erro, mas `np.interp` não confere a ordem e devolve um resultado sem sentido, **sem aviso**:

In [ ]:
print("np.interp com os dados na ordem do ensaio:", np.interp([30.0, 55.0, 85.0], T_ensaio, mu_ensaio).round(1))
print("np.interp com os dados ordenados:         ", np.interp([30.0, 55.0, 85.0], T_ensaio[ordem], mu_ensaio[ordem]).round(1))

É o tipo de erro que um teste com dados de exemplo bem comportados nunca encontra. Leia a documentação das funções que o agente escolhe, e teste com os dados reais.

## Resumo

| Método | Quando usar | Cuidado |
| --- | --- | --- |
| Polinômio (Lagrange baricêntrica, Newton) | poucos pontos, ou nós que você escolhe | grau alto com nós dados: Runge |
| Nós de Chebyshev | você escolhe onde avaliar uma função cara | é preciso poder escolher os nós |
| Spline cúbica | dados tabelados ou medidos | ordenar os dados; não extrapolar |
| PCHIP | dados que devem manter a monotonicidade | só derivada primeira contínua |
| Ajuste (cap. 7) | dados com ruído | escolher o modelo |

| Fórmula | Ideia |
| --- | --- |
| $f - p_n = \dfrac{f^{(n+1)}(\xi)}{(n+1)!}\prod(x - x_i)$ | erro = função × posição dos nós |
| $\max\lvert\prod(x - x_k)\rvert = 2^{-n}$ com Chebyshev | os melhores nós possíveis em $[-1, 1]$ |
| erro da spline cúbica $= O(h^4)$ | refinar a tabela funciona |

## Exercícios

**Conceituais (sem computador)**

1. Escreva os polinômios de Lagrange $\ell_0, \ell_1, \ell_2$ para os nós $0, 1, 3$ e mostre que $\ell_0 + \ell_1 + \ell_2 = 1$ para todo $x$. Por que isso vale para quaisquer nós?
2. Use {eq}`eq-erro-interp` para limitar o erro da interpolação linear de $\sin x$ com nós a cada $0{,}1$ rad. Quantos nós são necessários para garantir 6 algarismos?
3. Acrescente o ponto $(80;\ 0{,}355)$ à tabela da seção 4 e calcule a nova diferença dividida de 3ª ordem. Qual é a nova estimativa de $\mu(35)$?

**Verificação (com agente)**

4. Peça a um agente uma implementação da spline cúbica **presa** (derivadas dadas nas pontas). Verifique que ela reproduz exatamente qualquer polinômio de grau 3 e meça a ordem de convergência do erro com $\sin x$.
5. Peça a um agente a interpolação 2D de uma tabela de propriedades de vapor $h(p, T)$. Proponha dois invariantes físicos e um teste de validação cruzada para verificar o código.

**Projeto aberto**

6. **Trajetória de um braço robótico.** O braço deve passar por 6 pontos no plano, partindo e chegando parado. Compare uma spline cúbica presa (velocidade zero nas pontas) com um polinômio de grau 7. Desenhe a velocidade e a aceleração ao longo do caminho e discuta qual exige menos dos motores.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulo 18.
- L. N. Trefethen, *Approximation Theory and Approximation Practice*, SIAM, 2019, capítulos 1 a 5 e 13 a 15 (Chebyshev e o fenômeno de Runge).
- J.-P. Berrut e L. N. Trefethen, "Barycentric Lagrange interpolation", *SIAM Review* 46(3), 2004.